In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, round, when, current_date, current_timestamp, date_format, coalesce, lit
)

# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

# Read Raw Data from Lakehouse Files (Primary Dataset)
input_path = "Files/raw_data/dummyjson_products.csv"
raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(input_path)

# Data Cleaning 
cleaned_df = raw_df \
    .na.fill({
        "brand": "Unknown",
        "warrantyInformation": "No Warranty",
        "returnPolicy": "No Return Policy",
        "shippingInformation": "Standard Shipping"
    }) \
    .na.fill(0, subset=["price", "discountPercentage", "rating", "stock", "weight"])

# Format Date & Add Timestamp Columns
df_with_dates = cleaned_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("ingestion_date", date_format(current_date(), "yyyy-MM-dd"))

# Data Transformations 
transformed_df = df_with_dates \
    .withColumnRenamed("discountPercentage", "discount_percentage") \
    .withColumnRenamed("dimensions_width", "dimension_width") \
    .withColumnRenamed("dimensions_height", "dimension_height") \
    .withColumnRenamed("dimensions_depth", "dimension_depth") \
    .withColumnRenamed("warrantyInformation", "warranty_info") \
    .withColumnRenamed("shippingInformation", "shipping_info") \
    .withColumnRenamed("availabilityStatus", "availability_status") \
    .withColumnRenamed("returnPolicy", "return_policy") \
    .withColumnRenamed("minimumOrderQuantity", "min_order_qty") \
    .withColumn(
        "discounted_price", 
        round(col("price") * (1 - (col("discount_percentage") / 100)), 2)
    ) \
    .withColumn(
        "total_inventory_value", 
        round(col("price") * col("stock"), 2)
    ) \
    .withColumn(
        "rating_tier",
        when(col("rating") >= 4.5, "Top Rated")
        .when(col("rating") >= 4.0, "High Rated")
        .when(col("rating") >= 3.0, "Average")
        .otherwise("Low Rated")
    )

# Save as Lakehouse Delta Table 
transformed_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_products")

# Write transformed data back to Files directory 
transformed_df.write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save("Files/transformed_dummyjson_products")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, current_timestamp

spark = SparkSession.builder.getOrCreate()

# Ingest secondary dataset from Lakehouse Files/raw_data/
secondary_path = "Files/raw_data/category_product.txt"
raw_categories_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(secondary_path)

# Clean and transform data
dim_category_df = raw_categories_df \
    .na.fill({
        "supplier_region": "Global",
        "category_manager": "Unassigned"
    }) \
    .withColumn("vat_tax_pct", round(col("vat_tax_rate") * 100, 2)) \
    .withColumn("last_updated", current_timestamp())

# Save as secondary Delta Table
dim_category_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_category_details")

